# Pretraining for ASR

In [4]:
# installing libs
# !pip3 install torch torchvision torchaudio datasets transformers soundfile jiwer --index-url https://download.pytorch.org/whl/cu118
# !pip3 install librosa --index-url https://pypi.org/simple

In [29]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "3"
#os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import torch
import torch.nn as nn
import numpy as np

from datasets import load_dataset, disable_caching
import evaluate
from transformers import Wav2Vec2ForPreTraining, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Encoder

## Finetuning Wav2Vec2 model on CTC loss (5 points)


In this task you have to create pipeline for finetuning pretrained multilingual Wav2Vec2 model on belarusian audio from [Fleurs](https://huggingface.co/datasets/google/fleurs) dataset.

#### Prepare data

In [30]:
fleurs = load_dataset("google/fleurs", "be_by", split=["train", "validation", "test"], trust_remote_code=True)

In [7]:
fleurs[0][0]

{'id': 396,
 'num_samples': 250560,
 'path': 'C:\\Users\\nazmievairat\\.cache\\huggingface\\datasets\\downloads\\extracted\\b9db73aab3e937d51d5365ea33dc1771b3225fb604c206fbc10f661b62d7f956\\10009414287632395082.wav',
 'audio': {'path': 'train/10009414287632395082.wav',
  'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00031281,
         -0.00038069, -0.00132966]),
  'sampling_rate': 16000},
 'transcription': 'у той жа час паблізу ад верагодных маршрутаў уварвання базіравалася вельмі мала караблёў каралеўскага флоту таму што адміралы асцерагаліся іх патаплення нямецкімі паветранымі сіламі',
 'raw_transcription': 'У той жа час паблізу ад верагодных маршрутаў уварвання базіравалася вельмі мала караблёў каралеўскага флоту, таму што адміралы асцерагаліся іх патаплення нямецкімі паветранымі сіламі.',
 'gender': 1,
 'lang_id': 6,
 'language': 'Belarusian',
 'lang_group_id': 1}

In [8]:
fleurs[0]["transcription"][9]

'вышыня двух пілонаў складае 83 метры даўжыня моста - 378 метраў праезная частка складаецца з дзвюх палос шырыня кожнай - 3,50 м'

In this task, you should:

* filter all samples, where `transcription` includes digits. Hint: take care of specific belarussian symbols "і", "ў";
* remove punctuation from `transcription`.

In [9]:
def preprocess_dataset(example: dict) -> dict:
    example['transcription'] = re.sub(r"[^\w\s]", "", example['transcription'])
    return example

def filter_digits(example: dict) -> bool:
    return not bool(re.search(r"\d", example['transcription']))

In [10]:
preprocessed_train = fleurs[0].filter(filter_digits).map(preprocess_dataset)
preprocessed_val = fleurs[1].filter(filter_digits).map(preprocess_dataset)

#### Train tokenizer

There you should train your own BPE tokenizer based on texts from Fleurs dataset using [HuggingFace tokenizer](https://huggingface.co/docs/tokenizers/en/training_from_memory).

In [11]:
from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, decoders

# Define special tokens
PAD_TOKEN = "[PAD]"
BOS_TOKEN = "[BOS]"
EOS_TOKEN = "[EOS]"
UNK_TOKEN = "[UNK]"
VOCAB_SIZE = 1000

bpe_model = models.BPE(unk_token=UNK_TOKEN)

tokenizer = Tokenizer(bpe_model)

tokenizer.normalizer = normalizers.Sequence([
    normalizers.Lowercase(),
    #normalizers.Strip()
])

#tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

tokenizer.decoder = decoders.BPEDecoder()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=[PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
)

tokenizer.train_from_iterator(
    iterator=preprocessed_train['transcription'], 
    trainer=trainer
)

tokenizer.save("bpe_tokenizer.json")

#### Loading model and preprocessor

In [12]:
from transformers import Wav2Vec2FeatureExtractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
   "facebook/wav2vec2-xls-r-300m"
)
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m", 
    ctc_loss_reduction="mean", 
    pad_token_id=tokenizer.token_to_id(PAD_TOKEN),
    vocab_size=tokenizer.get_vocab_size(),
)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
feature_extractor

Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

#### Data processor and data collator 

In [ ]:

import copy

class CtcDataProcessor:
    def __init__(self, tokenizer, feature_extractor):
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor

    def __call__(self, row):
        """
            Function applies tokenizer on row['transcription'] and applies feature extractor on audio column in row.
            Input: dict with transcription and audio fields
            Output: original dict includes `labels` column with tokenized sequence and `input_values` column with computed spectrogram.
        """
        transcription = row['transcription']
        audio = row['audio']
        
        row['labels'] = self.tokenizer.encode(transcription).ids
        row["input_values"] = self.feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
        
        return row

In [15]:
data_processor = CtcDataProcessor(tokenizer, feature_extractor)
train = preprocessed_train.map(data_processor, keep_in_memory=True, remove_columns=preprocessed_train.column_names)
val = preprocessed_val.map(data_processor, keep_in_memory=True, remove_columns=preprocessed_val.column_names)

Map:   0%|          | 0/1927 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

In [35]:
single_audio = preprocessed_train.select([0])

In [19]:
from torch.nn.utils.rnn import pad_sequence

class CTCDataCollator:
    # HuggingFace requires pad transcript tokens with this value
    LABELS_PAD_IDX = -100

    @staticmethod
    def collate_tokens(tokens_batch: list, type: str, pad_value=0.0):
        """
        Collates a list of tokens (sequences) by padding them to the maximum length.
        For input_values, we use pad_value (default 0.0).
        For labels, we use LABELS_PAD_IDX.
        """
        tensors = [torch.tensor(tokens) for tokens in tokens_batch]
        if type == "input_values":
            return pad_sequence(tensors, batch_first=True, padding_value=pad_value)
        elif type == "labels":
            return pad_sequence(tensors, batch_first=True, padding_value=CTCDataCollator.LABELS_PAD_IDX)
        else:
            raise ValueError("Type must be either 'input_values' or 'labels'.")

    def __call__(self, batch):
        """
        Collates a list of dicts (from data processor) into one batch dict.
        Pads both input_values and labels sequences.
        Also creates an attention mask for the input_values (1 for real tokens, 0 for padded tokens).
        """
        input_values = [item["input_values"] for item in batch]
        labels = [item["labels"] for item in batch]

        input_values_padded = self.collate_tokens(input_values, type="input_values", pad_value=0.0)
        labels_padded = self.collate_tokens(labels, type="labels")


        attention_mask = (input_values_padded != 0.0).long()
        
        batch_collated = {
            "input_values": input_values_padded,
            "labels": labels_padded,
            "attention_mask": attention_mask,
        }

        return batch_collated

#### Inference and metrics computing

There you should use simple greedy straregy for CTC output decoding. 

Hint: Don't forget about padding value -100 in reference.

Hint: Don't forget about CTC output format.

In [20]:
from itertools import groupby
wer_metric = evaluate.load("wer")

def ctc_greedy_decoder(logits, blank_id):
    pred_ids = np.argmax(logits, axis=-1)
    decoded = []
    for seq in pred_ids:
        new_seq = []
        previous = None
        for token in seq:
            if token == blank_id:
                token = None
            if token is not None and token != previous:
                new_seq.append(token)
            previous = token
        decoded.append(new_seq)
    return decoded

class MetricsComputer:
    def __call__(self, pred):
        """
            Input: object with fields `predictions` for CTC model output and `label_ids` for tokenized reference;
            Output: dict with key `wer` and computed wer
        """
        # model prediction tensor, tensor batch_size x max_seq_len x vocab_size
        preds_logits = pred.predictions
        # reference, tensor batch_size x max_seq_len
        label_ids = pred.label_ids
        
        blank_id = tokenizer.token_to_id(PAD_TOKEN)
        
        decoded_preds = ctc_greedy_decoder(preds_logits, blank_id)
        pred_str = [tokenizer.decode(seq) for seq in decoded_preds]
        
        processed_labels = []
        for label in label_ids:
            filtered = [token for token in label if token != CTCDataCollator.LABELS_PAD_IDX]
            processed_labels.append(filtered)

        label_str = [tokenizer.decode(seq) for seq in processed_labels]
    
        print(f"Prediction: {pred_str[0]}")
        print(f"Reference: {label_str[0]}")
        
        wer = wer_metric.compute(predictions=pred_str, references=label_str)
        
        return {"wer": wer}

#### Overfitting on train batch

In this task you should check pipeline correctness by overfitting on you need to finetune Wav2Vec2 model and achieve 50 WER or lower accuracy on val set.

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="test",
    per_device_train_batch_size=2, # you could increase batch size
    gradient_accumulation_steps=8, 
    eval_strategy="steps",
    max_steps=1000,#3000,
    fp16=True,
    save_steps=50,
    eval_steps=10,
    logging_steps=10,
    learning_rate=1e-3,#1e-4,
    weight_decay=1e-5,
    warmup_steps=300,
    gradient_checkpointing=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=CTCDataCollator(),
    args=training_args,
    compute_metrics=MetricsComputer(),
    train_dataset=train,
    eval_dataset=val,
)

In [ ]:
train

Dataset({
    features: ['labels', 'input_values'],
    num_rows: 1927
})

In [24]:
feature_extractor(CTCDataCollator()([train[0], train[1]])['input_values'])

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


{'input_values': [array([[ 2.1443483e-04,  2.1443483e-04,  2.1443483e-04, ...,
        -9.1709225e-03, -1.1207873e-02, -3.9680481e-02],
       [ 2.8157950e-04,  2.8157950e-04,  2.8157950e-04, ...,
         7.0339459e-09,  7.0339459e-09,  7.0339459e-09]], dtype=float32)], 'attention_mask': [array([1, 1], dtype=int32)]}

In [25]:
model = model.cpu()

In [26]:
model(CTCDataCollator()([train[0], train[1]])['input_values'])

CausalLMOutput(loss=None, logits=tensor([[[-0.0561,  0.1173, -0.0424,  ..., -0.0726,  0.0542, -0.0705],
         [ 0.0551,  0.1438,  0.0242,  ...,  0.0713,  0.0823,  0.1038],
         [ 0.0484,  0.1405,  0.0195,  ...,  0.0740,  0.0706,  0.1150],
         ...,
         [ 0.0508,  0.1146,  0.0276,  ...,  0.0543,  0.0787,  0.1383],
         [ 0.0634,  0.1227,  0.0256,  ...,  0.0609,  0.0883,  0.1457],
         [ 0.0749,  0.0717,  0.0174,  ...,  0.0048,  0.1272,  0.0827]],

        [[ 0.0146,  0.1005, -0.0685,  ..., -0.0694,  0.0804, -0.0431],
         [ 0.0428,  0.1302,  0.0004,  ...,  0.0704,  0.0837,  0.1148],
         [ 0.0418,  0.1338,  0.0055,  ...,  0.0633,  0.0849,  0.1210],
         ...,
         [ 0.0358,  0.1275,  0.0039,  ...,  0.0816,  0.0726,  0.1340],
         [ 0.0347,  0.1311, -0.0023,  ...,  0.0748,  0.0777,  0.1354],
         [ 0.0389,  0.0628, -0.0169,  ..., -0.0113,  0.1110,  0.0338]]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [ ]:
trainer.train()